# Análisis exploratorio: `temp_data`

EDA reproducible de `temp_data/masked` y `temp_data/non_masked` (MRI + SPECT) para clasificación, generación y fusión con datos clínicos tabulares.

In [ ]:
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path("/workspaces/IL-CLDM")
TEMP = ROOT / "temp_data"
INFO_CSV = ROOT / "data_info" / "data_info.csv"
OUT_DIR = TEMP / "analysis_report"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# IL-CLDM crop (dataset.py)
CROP_SLICES = (slice(10, 170), slice(18, 210), slice(10, 170))
CROP_SHAPE = (160, 192, 160)

LABEL_NAMES = {0: "control", 1: "parkinson"}


def extract_id(path: Path, variant: str) -> str:
    if variant == "masked":
        return path.name.replace(".nii.gz", "").replace(".nii", "")
    m = re.match(r"(\d+)_", path.name)
    if not m:
        raise ValueError(f"Cannot parse ID from {path}")
    return m.group(1)


def infer_label_from_path(path: Path) -> int:
    p = str(path).lower()
    if "/control/" in p or "_control_" in path.name:
        return 0
    if "/parkinson/" in p or "_parkinson_" in path.name:
        return 1
    raise ValueError(f"Cannot infer label from {path}")


def load_volume(path, dtype=np.float32):
    return nib.load(str(path)).get_fdata().astype(dtype)


def brain_mask(data, threshold=0.0):
    return data > threshold


def volume_percentiles(data, qs=(1, 5, 25, 50, 75, 95, 99)):
    vals = data.ravel()
    if vals.size == 0:
        return {f"p{q}": np.nan for q in qs}
    return {f"p{q}": float(np.percentile(vals, q)) for q in qs}


def compute_intensity_stats(data):
    flat = data.ravel()
    mask = brain_mask(data)
    in_brain = data[mask] if mask.any() else flat
    pct = volume_percentiles(flat)
    out = {
        "min": float(np.min(flat)),
        "max": float(np.max(flat)),
        "mean": float(np.mean(flat)),
        "std": float(np.std(flat)),
        "median": float(np.median(flat)),
        "pct_zero": float((flat == 0).mean() * 100),
        "pct_negative": float((flat < 0).mean() * 100),
        "pct_nan": float(np.isnan(flat).mean() * 100),
        "pct_inf": float(np.isinf(flat).mean() * 100),
        "mean_in_brain": float(np.mean(in_brain)) if in_brain.size else np.nan,
        "std_in_brain": float(np.std(in_brain)) if in_brain.size else np.nan,
    }
    out.update(pct)
    return out


print(f"ROOT={ROOT}")
print(f"OUT_DIR={OUT_DIR}")


## 1. Inventario unificado

In [ ]:
def build_catalog():
    rows = []
    masked_root = TEMP / "masked"
    for modality, sub in [("MRI", "whole_MRI"), ("SPECT", "whole_SPECT")]:
        folder = masked_root / sub
        for p in sorted(folder.glob("*.nii*")):
            rows.append({
                "ID": extract_id(p, "masked"),
                "variant": "masked",
                "modality": modality,
                "path": str(p),
                "filename": p.name,
                "file_size_mb": p.stat().st_size / 1e6,
            })
    for p in sorted((TEMP / "non_masked").rglob("*.nii.gz")):
        modality = "MRI" if "/mri/" in str(p).lower() else "SPECT"
        rows.append({
            "ID": extract_id(p, "non_masked"),
            "variant": "non_masked",
            "modality": modality,
            "label_from_path": infer_label_from_path(p),
            "path": str(p),
            "filename": p.name,
            "file_size_mb": p.stat().st_size / 1e6,
        })
    catalog = pd.DataFrame(rows)
    info = pd.read_csv(INFO_CSV, dtype={"ID": str})
    catalog = catalog.merge(info[["ID", "label", "split"]], on="ID", how="left")
    if "label_from_path" in catalog.columns:
        mismatch = catalog[catalog["label"] != catalog["label_from_path"]]
        if len(mismatch):
            print("WARNING: label path vs CSV mismatch:", len(mismatch))
        catalog = catalog.drop(columns=["label_from_path"])
    return catalog, info


catalog, info_df = build_catalog()
catalog.to_csv(OUT_DIR / "catalog.csv", index=False)

# Pairing integrity
pair_checks = []
for (variant, modality), g in catalog.groupby(["variant", "modality"]):
    ids = set(g["ID"])
    pair_checks.append({"variant": variant, "modality": modality, "n_files": len(g), "n_unique_ids": g["ID"].nunique()})

pairs = catalog.pivot_table(index=["ID", "variant"], columns="modality", values="path", aggfunc="first").reset_index()
missing_mri = pairs["MRI"].isna().sum() if "MRI" in pairs else 0
missing_spect = pairs["SPECT"].isna().sum() if "SPECT" in pairs else 0

print(f"Catalog: {len(catalog)} files, {catalog['ID'].nunique()} unique IDs")
print(f"Variants: {catalog['variant'].value_counts().to_dict()}")
print(f"Modalities: {catalog['modality'].value_counts().to_dict()}")
print(f"Missing MRI pairs: {missing_mri}, Missing SPECT pairs: {missing_spect}")
display(catalog.head(8))


## 2. Resumen de cohorte

In [ ]:
cohort = (
    catalog.groupby(["variant", "modality", "label", "split"], dropna=False)
    .size()
    .reset_index(name="count")
)
cohort["label_name"] = cohort["label"].map(LABEL_NAMES)
cohort.to_csv(OUT_DIR / "cohort_counts.csv", index=False)

balance = (
    catalog.drop_duplicates(["ID", "variant"])
    .groupby(["variant", "label"])
    .size()
    .reset_index(name="n_subjects")
)
for variant in balance["variant"].unique():
    sub = balance[balance["variant"] == variant]
    c0 = sub.loc[sub["label"] == 0, "n_subjects"].sum() if (sub["label"] == 0).any() else 0
    c1 = sub.loc[sub["label"] == 1, "n_subjects"].sum() if (sub["label"] == 1).any() else 0
    ratio = c1 / c0 if c0 else np.nan
    print(f"{variant}: control={c0}, parkinson={c1}, ratio={ratio:.2f}")

fig, ax = plt.subplots(figsize=(8, 4))
for i, variant in enumerate(["masked", "non_masked"]):
    sub = cohort[(cohort["variant"] == variant) & (cohort["modality"] == "MRI")]
    x = np.arange(2)
    counts = [sub.loc[sub["label"] == lb, "count"].sum() for lb in [0, 1]]
    ax.bar(x + i * 0.35, counts, width=0.35, label=variant)
ax.set_xticks([0.175, 1.175])
ax.set_xticklabels(["control", "parkinson"])
ax.set_ylabel("N subjects (MRI files)")
ax.set_title("Cohort balance by variant")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "cohort_balance.png", dpi=120)
plt.show()
display(cohort)


## 3. Auditoría de headers NIfTI

In [ ]:
def header_row(path):
    img = nib.load(str(path))
    hdr = img.header
    data = img.get_fdata()
    aff = img.affine
    return {
        "path": path,
        "dtype_on_disk": str(img.get_data_dtype()),
        "shape": str(img.shape),
        "pixdim": str(tuple(np.round(hdr.get_zooms()[:3], 4))),
        "qform_code": int(img.get_qform(coded=True)[1]),
        "sform_code": int(img.get_sform(coded=True)[1]),
        "affine_diag": str(tuple(np.round(np.diag(aff)[:3], 4))),
        "scl_slope": float(hdr["scl_slope"]) if np.isfinite(hdr["scl_slope"]) else np.nan,
        "scl_inter": float(hdr["scl_inter"]) if np.isfinite(hdr["scl_inter"]) else np.nan,
        "has_nan": bool(np.isnan(data).any()),
        "has_inf": bool(np.isinf(data).any()),
    }


header_rows = []
for (variant, modality), g in catalog.groupby(["variant", "modality"]):
    sample_paths = g["path"].tolist()
    for p in sample_paths:
        row = header_row(p)
        row["variant"] = variant
        row["modality"] = modality
        header_rows.append(row)

header_df = pd.DataFrame(header_rows)
header_summary = (
    header_df.groupby(["variant", "modality"])
    .agg(
        n_files=("path", "count"),
        dtypes=("dtype_on_disk", lambda x: sorted(set(x))),
        shapes=("shape", lambda x: sorted(set(x))),
        pixdims=("pixdim", lambda x: sorted(set(x))),
        qform_codes=("qform_code", lambda x: sorted(set(x))),
        sform_codes=("sform_code", lambda x: sorted(set(x))),
        n_with_nan=("has_nan", "sum"),
        n_with_inf=("has_inf", "sum"),
    )
    .reset_index()
)
header_df.to_csv(OUT_DIR / "header_audit_per_file.csv", index=False)
header_summary.to_csv(OUT_DIR / "header_audit.csv", index=False)
print("Unique affines (masked MRI):", header_df[(header_df.variant == "masked") & (header_df.modality == "MRI")]["affine_diag"].nunique())
display(header_summary)


## 4. Geometría y crop IL-CLDM

In [ ]:
shape_counts = catalog.copy()
shape_rows = []
for _, row in catalog.iterrows():
    img = nib.load(row["path"])
    shape_rows.append({
        "ID": row["ID"],
        "variant": row["variant"],
        "modality": row["modality"],
        "shape": img.shape,
    })
shape_df = pd.DataFrame(shape_rows)
print("Unique shapes:", shape_df["shape"].value_counts().to_dict())

# MRI-SPECT shape match per subject
pivot_shape = shape_df.pivot_table(index=["ID", "variant"], columns="modality", values="shape", aggfunc="first")
mismatch = pivot_shape[pivot_shape["MRI"] != pivot_shape["SPECT"]] if "MRI" in pivot_shape and "SPECT" in pivot_shape else pd.DataFrame()
print(f"Shape mismatches MRI vs SPECT: {len(mismatch)}")

# Compare masked vs production data/
prod_mri = ROOT / "data" / "whole_MRI"
prod_abeta = ROOT / "data" / "whole_Abeta"
sample_id = catalog.loc[catalog.variant == "masked", "ID"].iloc[0]
if (prod_mri / f"{sample_id}.nii.gz").exists():
    m_temp = load_volume(TEMP / "masked" / "whole_MRI" / f"{sample_id}.nii.gz")
    m_prod = load_volume(prod_mri / f"{sample_id}.nii.gz")
    print(f"masked vs data/whole_MRI [{sample_id}]: max|diff|={np.max(np.abs(m_temp - m_prod)):.6f}")

# Crop visualization
example = catalog[(catalog.variant == "masked") & (catalog.modality == "MRI")].iloc[0]
vol = load_volume(example["path"])
z = vol.shape[2] // 2
sl = vol[:, :, z]
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(sl.T, cmap="gray", origin="lower")
# crop bounds in axial slice (x,y from slices 10:170 and 18:210)
from matplotlib.patches import Rectangle
rect = Rectangle((10, 18), 160, 192, linewidth=2, edgecolor="lime", facecolor="none")
ax.add_patch(rect)
ax.set_title(f"IL-CLDM crop on axial z={z} (ID {example['ID']})")
plt.tight_layout()
plt.savefig(FIG_DIR / "crop_bbox.png", dpi=120)
plt.show()

cropped = vol[CROP_SLICES]
print(f"Full shape {vol.shape} -> crop {cropped.shape} (expected {CROP_SHAPE})")
print(f"Voxels outside crop but >0: {((vol > 0) & (np.ones_like(vol, dtype=bool))).__class__}")
outside = vol.copy()
outside[CROP_SLICES] = 0
n_outside_brain = int(((outside > 0)).sum())
print(f"Non-zero voxels outside crop ROI: {n_outside_brain}")


## 5. Estadísticas de intensidad

In [ ]:
intensity_rows = []
for _, row in catalog.iterrows():
    data = load_volume(row["path"])
    stats = compute_intensity_stats(data)
    stats.update({
        "ID": row["ID"],
        "variant": row["variant"],
        "modality": row["modality"],
        "label": row["label"],
        "split": row["split"],
    })
    intensity_rows.append(stats)

intensity_df = pd.DataFrame(intensity_rows)
intensity_df.to_csv(OUT_DIR / "intensity_per_subject.csv", index=False)

summary_cols = ["min", "max", "mean", "std", "median", "pct_zero", "pct_negative", "mean_in_brain", "std_in_brain"]
intensity_summary = (
    intensity_df.groupby(["variant", "modality", "label"])[summary_cols]
    .agg(["mean", "std", "min", "max"])
)
intensity_summary.to_csv(OUT_DIR / "intensity_summary.csv")

# Histograms
for variant in ["masked", "non_masked"]:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, modality in zip(axes, ["MRI", "SPECT"]):
        sub = intensity_df[(intensity_df.variant == variant) & (intensity_df.modality == modality)]
        for label, color in [(0, "steelblue"), (1, "coral")]:
            ids = sub.loc[sub.label == label, "ID"].head(15)
            vals = []
            for sid in ids:
                p = catalog[(catalog.ID == sid) & (catalog.variant == variant) & (catalog.modality == modality)].path.iloc[0]
                v = load_volume(p)
                m = brain_mask(v)
                if m.any():
                    vals.append(v[m])
            if vals:
                pooled = np.concatenate(vals)
                ax.hist(pooled, bins=80, density=True, alpha=0.5, color=color, label=LABEL_NAMES[label])
        ax.set_title(f"{variant} / {modality}")
        ax.legend()
        if modality == "SPECT" and variant == "non_masked":
            ax.set_yscale("log")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"hist_{variant}.png", dpi=120)
    plt.show()

# Heatmap medians
med = intensity_df.groupby(["variant", "modality", "label"])["median"].mean().unstack("label")
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(med.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(med.columns)))
ax.set_xticklabels([LABEL_NAMES.get(c, c) for c in med.columns])
ax.set_yticks(range(len(med.index)))
ax.set_yticklabels([f"{v}/{m}" for v, m in med.index])
plt.colorbar(im, ax=ax, label="median intensity")
ax.set_title("Median intensity by group")
plt.tight_layout()
plt.savefig(FIG_DIR / "heatmap_median.png", dpi=120)
plt.show()

display(intensity_summary)


## 6. Máscara, fondo y ceros

In [ ]:
zero_rows = []
for (vid, mod), g in catalog.groupby(["variant", "modality"]):
    for _, row in g.iterrows():
        d = load_volume(row["path"])
        zero_rows.append({
            "ID": row["ID"],
            "variant": vid,
            "modality": mod,
            "label": row["label"],
            "pct_zero": (d == 0).mean() * 100,
        })
zero_df = pd.DataFrame(zero_rows)
zero_pivot = zero_df.pivot_table(index=["ID", "variant", "label"], columns="modality", values="pct_zero")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, variant in zip(axes, ["masked", "non_masked"]):
    sub = zero_pivot.reset_index()
    sub = sub[sub["variant"] == variant]
    ax.scatter(sub["MRI"], sub["SPECT"], alpha=0.5, s=12)
    lim = max(sub["MRI"].max(), sub["SPECT"].max()) * 1.05
    ax.plot([0, lim], [0, lim], "k--", alpha=0.3)
    ax.set_xlabel("% zero MRI")
    ax.set_ylabel("% zero SPECT")
    ax.set_title(variant)
plt.tight_layout()
plt.savefig(FIG_DIR / "zero_pct_scatter.png", dpi=120)
plt.show()

# Mask overlap example
for label in [0, 1]:
    row = catalog[(catalog.variant == "masked") & (catalog.label == label) & (catalog.modality == "MRI")].iloc[0]
    sid = row["ID"]
    mri_p = catalog[(catalog.ID == sid) & (catalog.variant == "masked") & (catalog.modality == "MRI")].path.iloc[0]
    spect_p = catalog[(catalog.ID == sid) & (catalog.variant == "masked") & (catalog.modality == "SPECT")].path.iloc[0]
    mri = load_volume(mri_p)
    spect = load_volume(spect_p)
    z = mri.shape[2] // 2
    union = (mri[:, :, z] > 0) | (spect[:, :, z] > 0)
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(union.T, cmap="gray", origin="lower")
    ax.set_title(f"Mask union axial z={z} | {LABEL_NAMES[label]} ID={sid}")
    plt.savefig(FIG_DIR / f"mask_union_label{label}.png", dpi=100)
    plt.close()

print(zero_df.groupby(["variant", "modality"])["pct_zero"].agg(["mean", "std"]).round(2))


## 7. Relación masked vs non_masked

In [ ]:
def pearson_on_mask(a, b, mask):
    av = a[mask].ravel()
    bv = b[mask].ravel()
    if av.size < 10:
        return np.nan
    return float(np.corrcoef(av, bv)[0, 1])


compare_rows = []
for modality in ["MRI", "SPECT"]:
    ids = catalog[catalog.variant == "masked"]["ID"].unique()
    for sid in ids:
        m_path = catalog[(catalog.ID == sid) & (catalog.variant == "masked") & (catalog.modality == modality)]
        n_path = catalog[(catalog.ID == sid) & (catalog.variant == "non_masked") & (catalog.modality == modality)]
        if m_path.empty or n_path.empty:
            continue
        masked_v = load_volume(m_path.path.iloc[0])
        raw_v = load_volume(n_path.path.iloc[0])
        mask = brain_mask(raw_v)
        compare_rows.append({
            "ID": sid,
            "modality": modality,
            "mean_masked": float(masked_v.mean()),
            "mean_raw": float(raw_v.mean()),
            "std_masked": float(masked_v.std()),
            "std_raw": float(raw_v.std()),
            "corr_brain": pearson_on_mask(masked_v, raw_v, mask),
        })

compare_df = pd.DataFrame(compare_rows)
compare_df.to_csv(OUT_DIR / "masked_vs_non_masked.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, mod in zip(axes, ["MRI", "SPECT"]):
    sub = compare_df[compare_df.modality == mod]
    ax.scatter(sub["mean_raw"], sub["mean_masked"], alpha=0.4, s=10)
    ax.set_xlabel("mean non_masked")
    ax.set_ylabel("mean masked")
    ax.set_title(f"{mod} (r brain: {sub['corr_brain'].mean():.3f})")
plt.tight_layout()
plt.savefig(FIG_DIR / "masked_vs_raw_mean.png", dpi=120)
plt.show()
display(compare_df.groupby("modality")[["corr_brain", "mean_masked", "mean_raw"]].mean().round(4))


## 8. Codificación int16 (SPECT non_masked)

In [ ]:
spect_nm = catalog[(catalog.variant == "non_masked") & (catalog.modality == "SPECT")]
int16_stats = []
for _, row in spect_nm.iterrows():
    img = nib.load(row["path"])
    raw = np.asanyarray(img.dataobj)
    scaled = img.get_fdata()
    int16_stats.append({
        "ID": row["ID"],
        "dtype_disk": str(img.get_data_dtype()),
        "raw_min": float(np.min(raw)),
        "raw_max": float(np.max(raw)),
        "scaled_min": float(np.min(scaled)),
        "scaled_max": float(np.max(scaled)),
        "near_int16_max": float((raw > 30000).mean() * 100),
    })
int16_df = pd.DataFrame(int16_stats)
int16_df.to_csv(OUT_DIR / "spect_int16_audit.csv", index=False)

example_path = spect_nm.path.iloc[0]
img = nib.load(example_path)
raw = np.asanyarray(img.dataobj).ravel()
scaled = img.get_fdata().ravel()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(raw, bins=80, color="steelblue", log=True)
axes[0].set_title("SPECT non_masked: valores en disco (int16)")
axes[1].hist(scaled, bins=80, color="coral", log=True)
axes[1].set_title("Tras get_fdata()")
plt.tight_layout()
plt.savefig(FIG_DIR / "spect_int16_hist.png", dpi=120)
plt.show()
print(int16_df.describe().round(2))


## 9. QC visual estratificado

In [ ]:
def plot_pair(subject_id, variant, z=None, save=None):
    mri_row = catalog[(catalog.ID == subject_id) & (catalog.variant == variant) & (catalog.modality == "MRI")]
    spect_row = catalog[(catalog.ID == subject_id) & (catalog.variant == variant) & (catalog.modality == "SPECT")]
    if mri_row.empty or spect_row.empty:
        return
    mri = load_volume(mri_row.path.iloc[0])
    spect = load_volume(spect_row.path.iloc[0])
    if z is None:
        z = mri.shape[2] // 2
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(mri[:, :, z].T, cmap="gray", origin="lower")
    axes[0].set_title(f"MRI z={z}")
    axes[1].imshow(spect[:, :, z].T, cmap="gray", origin="lower")
    axes[1].set_title(f"SPECT z={z}")
    label = int(mri_row.label.iloc[0])
    fig.suptitle(f"{variant} | {LABEL_NAMES[label]} | ID={subject_id}")
    plt.tight_layout()
    if save:
        plt.savefig(save, dpi=100)
        plt.close()
    else:
        plt.show()


qc_ids = []
for variant in ["masked", "non_masked"]:
    for label in [0, 1]:
        sub = catalog[(catalog.variant == variant) & (catalog.label == label) & (catalog.modality == "MRI")]
        if not sub.empty:
            sid = sub.iloc[0]["ID"]
            qc_ids.append((variant, label, sid))
            plot_pair(sid, variant, save=FIG_DIR / f"qc_{variant}_{LABEL_NAMES[label]}_{sid}.png")

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for ax, (variant, label, sid) in zip(axes.ravel(), qc_ids):
    mri_row = catalog[(catalog.ID == sid) & (catalog.variant == variant) & (catalog.modality == "MRI")]
    mri = load_volume(mri_row.path.iloc[0])
    z = mri.shape[2] // 2
    ax.imshow(mri[:, :, z].T, cmap="gray", origin="lower")
    ax.set_title(f"{variant}\n{LABEL_NAMES[label]} {sid}")
    ax.axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "qc_grid_mri.png", dpi=120)
plt.show()
print("QC figures saved to", FIG_DIR)



## 10. Recomendaciones para modelos de IA

### Clasificación (por modalidad)
- Entrenar **encoders separados** para MRI y SPECT; no compartir normalización entre modalidades.
- Preferir **`temp_data/masked`** (equivalente a `data/whole_MRI` y pipeline IL-CLDM): intensidades ya normalizadas (~media 0, σ≈0.5, fondo en cero).
- Aplicar siempre el **mismo crop** `[10:170, 18:210, 10:170]` → `(160, 192, 160)` como en `dataset.py`.
- Cohorte **desbalanceada** (114 parkinson / 73 control): usar class weights, splits estratificados desde `data_info/`, y métricas `balanced_accuracy` / ROC-AUC.
- Normalización única: z-score por volumen en `dataset.nifti_to_numpy` (no hay opción min-max).

### Tareas generativas (MRI → SPECT)
- Usar pareamiento estricto por **ID** (187 pares completos en ambas variantes).
- Target recomendado: **`masked/whole_SPECT`** (análogo a `data/whole_Abeta` en producción).
- **`non_masked`** requiere pipeline explícito: `int16` → `float32`, revisar `scl_slope/inter`, y normalización por volumen o por cohorte antes del AAE/LDM.
- Verificar que la **máscara de fondo** sea coherente entre MRI condicionante y SPECT objetivo (en masked, ~71% ceros y alineados entre modalidades).

### Integración con datos clínicos tabulares
- Clave de join: **`ID` (str)**; usar `analysis_report/catalog.csv` como tabla puente.
- Registrar metadatos obligatorios: `variant`, `preprocessing_version`, `modality`, `split`.
- Evitar leakage: estadísticas de normalización tabular/imagen solo desde **train**.
- Arquitectura sugerida: embeddings por modalidad + **late fusion** con MLP sobre features clínicos escalados.

### Checklist pre-entrenamiento
- [ ] Un solo pipeline de intensidad por experimento (no mezclar masked y non_masked)
- [ ] dtype homogéneo en disco y tras carga (`float32` recomendado)
- [ ] Shape única `(182,218,182)` pre-crop; `(160,192,160)` en modelo
- [ ] Splits fijos desde `data_info/`
- [ ] Revisar % ceros y valores negativos por modalidad
- [ ] SPECT non_masked: auditar saturación int16 y negativos tras `get_fdata()`

---
**Artefactos exportados:** `temp_data/analysis_report/` (`catalog.csv`, `intensity_*.csv`, `header_audit.csv`, `masked_vs_non_masked.csv`, `figures/`).


Temporal for revision of data

In [ ]:
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path



In [ ]:
from pathlib import Path

def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / "config.py").is_file():
            return parent
    raise FileNotFoundError("Could not find repo root (config.py)")

ROOT = find_repo_root()
SUBJECT_ID = "3102"
mri_path = ROOT / "dataV2" / "whole_MRI" / f"{SUBJECT_ID}.nii.gz"
mri = nib.load(mri_path)
mri_data = mri.get_fdata()

spect_path = ROOT / "dataV2" / "whole_Abeta" / f"{SUBJECT_ID}.nii.gz"
spect = nib.load(spect_path)
spect_data = spect.get_fdata()

print(mri_data.shape)
print(spect_data.shape)

#know the min and max values of the data
print("MRI")
print(np.min(mri_data))
print(np.max(mri_data))
print(np.mean(mri_data))
print(np.std(mri_data))
print("SPECT")
print(np.min(spect_data))
print(np.max(spect_data))
print(np.mean(spect_data))
print(np.std(spect_data))

In [ ]:
# Comparación de estadísticas MRI vs SPECT (whole_MRI vs whole_Abeta)
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from scipy import stats

# --- rutas ---
def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / "config.py").is_file():
            return parent
    raise FileNotFoundError("Could not find repo root (config.py)")

ROOT = find_repo_root()
DATA_ROOT = ROOT / "dataV2"        # cambiar a ROOT / "data" si quieres los symlinks
MRI_DIR = DATA_ROOT / "whole_MRI"
SPECT_DIR = DATA_ROOT / "whole_Abeta"   # SPECT DaT (nombre legacy whole_Abeta)
OUT_DIR = ROOT / "docs" / "reports" / "mri_spect_stats"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Opcional: mismo crop que dataset.py (None = volumen completo)
USE_CROP = False
CROP = (slice(10, 170), slice(18, 210), slice(10, 170))

def subject_id(path: Path) -> str:
    n = path.name
    return n[:-7] if n.endswith(".nii.gz") else n[:-4]

def load_vol(path: Path) -> np.ndarray:
    v = nib.load(str(path)).get_fdata(dtype=np.float32)
    if USE_CROP:
        v = v[CROP]
    return v

def vol_stats(v: np.ndarray, mask_positive: bool = True) -> dict:
    flat = v.ravel()
    if mask_positive:
        vals = flat[flat > 0]
        if vals.size == 0:
            vals = flat
    else:
        vals = flat
    return {
        "mean": float(np.mean(vals)),
        "std": float(np.std(vals)),
        "median": float(np.median(vals)),
        "p01": float(np.percentile(vals, 1)),
        "p99": float(np.percentile(vals, 99)),
        "min": float(np.min(flat)),
        "max": float(np.max(flat)),
        "pct_zero": float((flat == 0).mean() * 100),
        "n_voxels": int(flat.size),
        "n_brain": int(vals.size),
    }

# --- emparejar por ID ---
mri_files = {subject_id(p): p for p in sorted(MRI_DIR.glob("*.nii*"))}
spect_files = {subject_id(p): p for p in sorted(SPECT_DIR.glob("*.nii*"))}
common = sorted(set(mri_files) & set(spect_files))
only_mri = sorted(set(mri_files) - set(spect_files))
only_spect = sorted(set(spect_files) - set(mri_files))

print(f"MRI: {len(mri_files)} | SPECT: {len(spect_files)} | Pareados: {len(common)}")
if only_mri or only_spect:
    print(f"  Solo MRI: {len(only_mri)} | Solo SPECT: {len(only_spect)}")

rows = []
for sid in common:
    m = load_vol(mri_files[sid])
    s = load_vol(spect_files[sid])
    if m.shape != s.shape:
        rows.append({"ID": sid, "error": f"shape mismatch {m.shape} vs {s.shape}"})
        continue
    sm, ss = vol_stats(m), vol_stats(s)
    rows.append({
        "ID": sid,
        "shape": str(m.shape),
        "mri_mean": sm["mean"], "spect_mean": ss["mean"],
        "mri_std": sm["std"], "spect_std": ss["std"],
        "mri_median": sm["median"], "spect_median": ss["median"],
        "mri_p99": sm["p99"], "spect_p99": ss["p99"],
        "ratio_mean": ss["mean"] / sm["mean"] if sm["mean"] != 0 else np.nan,
        "ratio_std": ss["std"] / sm["std"] if sm["std"] != 0 else np.nan,
        "diff_mean": ss["mean"] - sm["mean"],
        "diff_std": ss["std"] - sm["std"],
        "abs_z_mean": abs(ss["mean"] - sm["mean"]) / max(sm["std"], 1e-8),  # escala relativa por sujeto
    })

df = pd.DataFrame([r for r in rows if "error" not in r])
err = pd.DataFrame([r for r in rows if "error" in r])
df.to_csv(OUT_DIR / "per_subject_mri_spect_stats.csv", index=False)
print(df.head(3))

In [ ]:
# --- resumen de cohorte: ¿están desfasadas? ---
def summarize(series: pd.Series):
    return pd.Series({
        "median": series.median(),
        "mean": series.mean(),
        "std": series.std(),
        "p05": series.quantile(0.05),
        "p95": series.quantile(0.95),
    })

metrics = ["mri_mean", "spect_mean", "mri_std", "spect_std", "ratio_mean", "ratio_std", "diff_mean", "abs_z_mean"]
summary = pd.DataFrame({m: summarize(df[m]) for m in metrics}).T
display(summary)

# Correlación entre medias por sujeto (alto = escalas covarian entre sujetos)
r_mean, p_mean = stats.pearsonr(df["mri_mean"], df["spect_mean"])
r_std, p_std = stats.pearsonr(df["mri_std"], df["spect_std"])
print(f"Correlación MRI_mean vs SPECT_mean: r={r_mean:.3f}, p={p_mean:.2e}")
print(f"Correlación MRI_std  vs SPECT_std : r={r_std:.3f}, p={p_std:.2e}")

# Test si las distribuciones de *medias por sujeto* difieren entre modalidades
ks_mean = stats.ks_2samp(df["mri_mean"], df["spect_mean"])
ks_std = stats.ks_2samp(df["mri_std"], df["spect_std"])
print(f"KS (medias por sujeto): stat={ks_mean.statistic:.3f}, p={ks_mean.pvalue:.2e}")
print(f"KS (std por sujeto)   : stat={ks_std.statistic:.3f}, p={ks_std.pvalue:.2e}")

# Cohen's d entre pooled per-subject means (tamaño del desfase global)
def cohens_d(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)

d_mean = cohens_d(df["mri_mean"], df["spect_mean"])
d_std = cohens_d(df["mri_std"], df["spect_std"])
print(f"Cohen's d (mean): {d_mean:.3f} | (std): {d_std:.3f}")

# Heurística simple de alineación (ajusta umbrales según tu criterio)
ratio_med = df["ratio_mean"].median()
ratio_iqr = df["ratio_mean"].quantile(0.75) - df["ratio_mean"].quantile(0.25)
flags = []
if abs(ratio_med - 1.0) > 0.5:
    flags.append(f"ratio_mean mediana={ratio_med:.2f} lejos de 1")
if ratio_iqr > 1.0:
    flags.append(f"ratio_mean IQR alto={ratio_iqr:.2f} (heterogeneidad entre sujetos)")
if r_mean < 0.5:
    flags.append(f"correlación baja entre medias (r={r_mean:.2f})")
if abs(d_mean) > 0.8:
    flags.append(f"Cohen's d grande en medias (|d|={abs(d_mean):.2f})")

print("\n--- Diagnóstico ---")
if not flags:
    print("OK: escalas por sujeto razonablemente alineadas (heurística).")
else:
    print("Posible desfase:")
    for f in flags:
        print(" -", f)

In [ ]:
# --- gráficos ---
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

axes[0, 0].scatter(df["mri_mean"], df["spect_mean"], alpha=0.5, s=20)
lims = [min(df["mri_mean"].min(), df["spect_mean"].min()),
        max(df["mri_mean"].max(), df["spect_mean"].max())]
axes[0, 0].plot(lims, lims, "r--", lw=1, label="y=x")
axes[0, 0].set_xlabel("MRI mean (>0 voxels)")
axes[0, 0].set_ylabel("SPECT mean")
axes[0, 0].set_title(f"Medias por sujeto (r={r_mean:.2f})")
axes[0, 0].legend()

axes[0, 1].hist(df["ratio_mean"], bins=30, edgecolor="k", alpha=0.7)
axes[0, 1].axvline(1.0, color="r", ls="--", label="ratio=1")
axes[0, 1].axvline(ratio_med, color="orange", ls="-", label=f"mediana={ratio_med:.2f}")
axes[0, 1].set_xlabel("SPECT_mean / MRI_mean")
axes[0, 1].set_title("Distribución del ratio de medias")

axes[1, 0].boxplot(
    [df["mri_mean"], df["spect_mean"]],
    tick_labels=["MRI", "SPECT"],
)
axes[1, 0].set_title("Boxplot de medias por sujeto")
axes[1, 0].set_ylabel("mean intensity")

axes[1, 1].hist(df["diff_mean"], bins=30, edgecolor="k", alpha=0.7)
axes[1, 1].axvline(0, color="r", ls="--")
axes[1, 1].set_xlabel("SPECT_mean - MRI_mean")
axes[1, 1].set_title("Diferencia de medias")

plt.suptitle(f"{DATA_ROOT.name}: whole_MRI vs whole_Abeta", y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "mri_spect_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- (opcional) comparar histogramas globales en un sujeto ejemplo ---
sid = common[0]
m = load_vol(mri_files[sid])
s = load_vol(spect_files[sid])

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for ax_i, vol, title in zip(ax, [m, s], ["MRI", "SPECT"]):
    vals = vol[vol > 0].ravel()
    ax_i.hist(vals, bins=80, density=True, alpha=0.8)
    ax_i.set_title(f"{title} ID={sid}")
    ax_i.set_xlabel("intensity")
fig.suptitle("Histogramas (voxels > 0) — un sujeto")
plt.tight_layout()
plt.show()

In [ ]:
import sys
from pathlib import Path

def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / "config.py").is_file():
            return parent
    raise FileNotFoundError("Could not find repo root (config.py)")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))
from viz_pairs import plot_subject

plot_subject(ROOT / "dataV2", "3102")

In [ ]:
#plot the image (optional local raw SPECT; adjust path if you have it)
path = ROOT / "temp_data" / "non_masked" / "spect" / f"{SUBJECT_ID}_spect_coregistered.nii.gz"
nii = nib.load(path)
data = nii.get_fdata()

#plot the image
plt.imshow(data[:, :, 100], cmap="gray")
plt.show()

#plot the histogram of the data
plt.hist(data.ravel(), bins=100, edgecolor="k", alpha=0.7)
plt.show()

### Temporal analisis to understand the different preprocessing steps applied along all the phases of the proyect

In [ ]:
import os, glob
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt

DIRS = {
    "MRI":   "/workspace/IL-CLDM/data/whole_MRI",
    "Abeta": "/workspace/IL-CLDM/data/whole_Abeta",
}

# Pon None para analizar TODOS los volúmenes (más lento). Un entero = muestra aleatoria.
N_SAMPLE = 30
RNG = np.random.default_rng(0)

In [ ]:
def volume_stats(path):
    """Estadísticas de un volumen. 'fg' = solo voxeles distintos de cero (foreground)."""
    vol = nib.load(path).get_fdata(dtype=np.float32)
    fg = vol[vol != 0]
    if fg.size == 0:
        fg = vol.ravel()
    return {
        "file": os.path.basename(path),
        "dtype": str(nib.load(path).get_data_dtype()),
        "shape": "x".join(map(str, vol.shape)),
        "min": float(vol.min()),
        "max": float(vol.max()),
        "mean": float(vol.mean()),
        "std": float(vol.std()),
        "fg_mean": float(fg.mean()),
        "fg_std": float(fg.std()),
        "pct_neg": float((vol < 0).mean() * 100),   # % de voxeles negativos
        "n_unique_le256": int(min(np.unique(vol).size, 257)),  # <=256 sugiere intensidades discretas/enteras
    }

records = {}
for name, d in DIRS.items():
    files = sorted(glob.glob(os.path.join(d, "*.nii.gz")))
    if N_SAMPLE is not None and len(files) > N_SAMPLE:
        files = list(RNG.choice(files, size=N_SAMPLE, replace=False))
    rows = [volume_stats(f) for f in files]
    records[name] = pd.DataFrame(rows)
    print(f"{name}: {len(rows)} volúmenes analizados de {d}")

In [ ]:
for name, df in records.items():
    print(f"\n===== {name} =====")
    agg = df[["min","max","mean","std","fg_mean","fg_std","pct_neg"]].agg(["mean","std","min","max"])
    print(agg.round(4).to_string())
    print("dtypes:", df["dtype"].value_counts().to_dict())
    print("shapes:", df["shape"].value_counts().to_dict())
    display(df.round(4).head(10))

In [ ]:
def diagnose(df, name):
    gmin, gmax = df["min"].min(), df["max"].max()
    mean_of_max = df["max"].mean()
    mean_mean, mean_std = df["mean"].mean(), df["std"].mean()
    fg_mean, fg_std = df["fg_mean"].mean(), df["fg_std"].mean()
    pct_neg = df["pct_neg"].mean()

    print(f"\n===== Veredicto {name} =====")
    print(f"rango global: [{gmin:.4f}, {gmax:.4f}] | max medio por vol: {mean_of_max:.4f}")
    print(f"media global: {mean_mean:.4f}  std global: {mean_std:.4f}")
    print(f"foreground -> media: {fg_mean:.4f}  std: {fg_std:.4f} | % voxeles negativos: {pct_neg:.2f}%")

    verdict = []
    # min-max a [0,1]
    if gmin >= -1e-3 and abs(gmax - 1.0) < 0.05 and abs(mean_of_max - 1.0) < 0.1:
        verdict.append("Min-Max a [0,1] (max≈1, min≈0)")
    # escalado a [-1,1]
    elif abs(gmin + 1.0) < 0.05 and abs(gmax - 1.0) < 0.05:
        verdict.append("Escalado a [-1,1]")
    # z-score (media≈0, std≈1)
    if abs(fg_mean) < 0.15 and abs(fg_std - 1.0) < 0.25:
        verdict.append("Z-score / estandarización (media≈0, std≈1 en foreground)")
    # rango fijo pero no unitario (p.ej. percentil/SUVR clip)
    if gmax > 1.5 and df["max"].std() / (df["max"].mean() + 1e-9) < 0.05:
        verdict.append(f"Rango fijo consistente (~{mean_of_max:.2f}), posible clip por percentil/SUVR")

    if not verdict:
        # datos crudos: los rangos suelen variar mucho entre volúmenes
        cv_max = df["max"].std() / (df["max"].mean() + 1e-9)
        if cv_max > 0.15:
            verdict.append("SIN normalización aparente: rangos variables entre volúmenes (datos crudos)")
        else:
            verdict.append("No concluyente: revisa histograma")

    for v in verdict:
        print("  ->", v)

for name, df in records.items():
    diagnose(df, name)

In [ ]:
fig, axes = plt.subplots(1, len(DIRS), figsize=(6*len(DIRS), 4))
if len(DIRS) == 1: axes = [axes]

for ax, (name, d) in zip(axes, DIRS.items()):
    f = sorted(glob.glob(os.path.join(d, "*.nii.gz")))[0]
    vol = nib.load(f).get_fdata(dtype=np.float32)
    fg = vol[vol != 0]
    ax.hist(fg, bins=100)
    ax.set_title(f"{name} — {os.path.basename(f)}\n[{vol.min():.3f}, {vol.max():.3f}] foreground")
    ax.set_xlabel("intensidad"); ax.set_ylabel("frecuencia")
plt.tight_layout(); plt.show()

## Construcción de `data_rois` usando ROI de ganglios basales

Este bloque crea una nueva versión de datos en `data_rois` a partir de `data/whole_MRI` y `data/whole_Abeta`, calculando un bounding box por caso desde la máscara de ganglios basales (`mask_basal_ganglia`) y expandiéndolo **15 vóxeles** en cada dirección. El contenido original en `data` no se modifica.

In [1]:
from pathlib import Path
import numpy as np
import nibabel as nib

ROOT = Path("/workspace/IL-CLDM")
DATA_IN = ROOT / "data"
DATA_OUT = ROOT / "data_rois"

MRI_IN = DATA_IN / "whole_MRI"
ABETA_IN = DATA_IN / "whole_Abeta"
MRI_OUT = DATA_OUT / "whole_MRI"
ABETA_OUT = DATA_OUT / "whole_Abeta"
MASK_OUT = DATA_OUT / "mask_basal_ganglia"

MARGIN = 15  # voxeles en cada dirección


# Se intenta detectar automáticamente la ubicación de máscaras.
MASK_CANDIDATES = [
    DATA_IN / "mask_basal_ganglia",
    DATA_IN / "mask_basal_ganglia.nii.gz",
    ROOT / "mask_basal_ganglia",
    ROOT / "mask_basal_ganglia.nii.gz",
]


def ensure_dirs():
    for p in [MRI_OUT, ABETA_OUT, MASK_OUT]:
        p.mkdir(parents=True, exist_ok=True)


def discover_case_ids(folder: Path):
    return {p.name.replace(".nii.gz", "") for p in folder.glob("*.nii.gz")}


def find_mask_source():
    existing = [p for p in MASK_CANDIDATES if p.exists()]
    if not existing:
        raise FileNotFoundError(
            "No se encontró la máscara `mask_basal_ganglia`. "
            "Revisa la ruta en MASK_CANDIDATES."
        )
    # Prioriza archivo único si existe; luego directorio.
    files = [p for p in existing if p.is_file()]
    if files:
        return files[0]
    return existing[0]


def load_case_mask(mask_source: Path, case_id: str):
    if mask_source.is_file():
        return nib.load(str(mask_source))

    # Si es directorio, intenta varios nombres comunes por caso.
    patterns = [
        f"{case_id}.nii.gz",
        f"{case_id}_mask_basal_ganglia.nii.gz",
        f"mask_basal_ganglia_{case_id}.nii.gz",
    ]
    for name in patterns:
        p = mask_source / name
        if p.exists():
            return nib.load(str(p))

    return None


def compute_bbox(mask_data: np.ndarray, margin: int = 15):
    coords = np.argwhere(mask_data > 0)
    if coords.size == 0:
        return None
    mins = coords.min(axis=0) - margin
    maxs = coords.max(axis=0) + margin + 1  # slice end exclusivo
    mins = np.maximum(mins, 0)
    maxs = np.minimum(maxs, np.array(mask_data.shape))
    return tuple(slice(int(lo), int(hi)) for lo, hi in zip(mins, maxs))


def crop_nifti(img: nib.Nifti1Image, bbox):
    data = img.get_fdata(dtype=np.float32)
    cropped = data[bbox]

    affine = img.affine.copy()
    start = np.array([bbox[0].start, bbox[1].start, bbox[2].start], dtype=np.float64)
    affine[:3, 3] = affine[:3, 3] + affine[:3, :3] @ start

    out = nib.Nifti1Image(cropped, affine, header=img.header.copy())
    out.set_data_dtype(np.float32)
    return out


def save_cropped(case_id: str, bbox, mask_img):
    mri_path = MRI_IN / f"{case_id}.nii.gz"
    abeta_path = ABETA_IN / f"{case_id}.nii.gz"

    if mri_path.exists():
        mri_img = nib.load(str(mri_path))
        nib.save(crop_nifti(mri_img, bbox), str(MRI_OUT / f"{case_id}.nii.gz"))

    if abeta_path.exists():
        abeta_img = nib.load(str(abeta_path))
        nib.save(crop_nifti(abeta_img, bbox), str(ABETA_OUT / f"{case_id}.nii.gz"))

    nib.save(crop_nifti(mask_img, bbox), str(MASK_OUT / f"{case_id}.nii.gz"))


ensure_dirs()
mask_source = find_mask_source()

ids = sorted(discover_case_ids(MRI_IN) | discover_case_ids(ABETA_IN))
processed, skipped = 0, []

for case_id in ids:
    mask_img = load_case_mask(mask_source, case_id)
    if mask_img is None:
        skipped.append((case_id, "sin máscara por caso"))
        continue

    mask_data = mask_img.get_fdata(dtype=np.float32)
    bbox = compute_bbox(mask_data, margin=MARGIN)
    if bbox is None:
        skipped.append((case_id, "máscara vacía"))
        continue

    save_cropped(case_id, bbox, mask_img)
    processed += 1

print(f"Máscara usada desde: {mask_source}")
print(f"Casos totales detectados: {len(ids)}")
print(f"Casos procesados: {processed}")
print(f"Casos omitidos: {len(skipped)}")
if skipped:
    print("Ejemplos omitidos:", skipped[:5])
print(f"Salida generada en: {DATA_OUT}")

Máscara usada desde: /workspace/IL-CLDM/data/mask_basal_ganglia.nii.gz
Casos totales detectados: 184
Casos procesados: 184
Casos omitidos: 0
Salida generada en: /workspace/IL-CLDM/data_rois


## Arquitectura AAE (dimensiones capa a capa)

Carga el checkpoint E1 MRI (`MRI_AAE.pth.tar`), instancia el `AAE` del proyecto y traza las formas de activación con un volumen dummy del tamaño de trabajo ROI (`1×108×104×104`).

In [10]:
import os
import sys
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn

ROOT = Path("/workspace/IL-CLDM")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Rutas relativas de config (data, masks, etc.) asumen cwd = raíz del repo.
os.chdir(ROOT)

# Jupyter precarga el módulo estándar `code`, que tapa el paquete local code/.
# Hay que expulsarlo antes de `import model` (model.py hace `from code.volume_shapes`).
sys.modules.pop("code", None)

import config
from model import AAE

model_path = ROOT / "result/exp_nm_s1_mri_rois_supcon/phase1_mri/checkpoints/MRI_AAE.pth.tar"
device = torch.device("cpu")

# El .pth.tar guarda un dict {"state_dict", "optimizer", "epoch", ...}, no el nn.Module.
payload = torch.load(model_path, map_location=device, weights_only=False)
state = payload["state_dict"] if isinstance(payload, dict) and "state_dict" in payload else payload

model = AAE().to(device)
missing, unexpected = model.load_state_dict(state, strict=False)
model.eval()
print(f"Checkpoint: {model_path}")
print(f"epoch={payload.get('epoch') if isinstance(payload, dict) else 'n/a'}  "
      f"selection_score={payload.get('selection_score') if isinstance(payload, dict) else 'n/a'}")
print(f"load_state_dict: missing={len(missing)} unexpected={len(unexpected)}")
print(f"config.latent_dim={config.latent_dim}  device={device}\n")

# Volumen de trabajo ROI (tras pad a múltiplos de 4): (B, C, D, H, W)
dummy = torch.zeros(1, 1, 108, 104, 104, device=device)

rows = []
hooks = []


def _shape(t):
    return tuple(int(x) for x in t.shape)


def make_hook(name):
    def hook(_module, _inp, out):
        if torch.is_tensor(out):
            rows.append({"stage": name, "shape": _shape(out), "numel": int(out.numel())})
        elif isinstance(out, (tuple, list)) and out and torch.is_tensor(out[0]):
            rows.append({"stage": name, "shape": _shape(out[0]), "numel": int(out[0].numel())})
    return hook


# Entrada + cada submódulo del encoder/decoder (Sequential interno).
rows.append({"stage": "input", "shape": _shape(dummy), "numel": int(dummy.numel())})
for name, module in model.named_modules():
    if name == "" or isinstance(module, (nn.Sequential, AAE)):
        continue
    # Solo capas “hoja” con cómputo (evitar dobles hooks en contenedores ResidualBlock enteros
    # registrando también el contenedor ayuda a ver el residual; lo dejamos).
    if any(isinstance(module, t) for t in (nn.Conv3d, nn.ConvTranspose3d, nn.ReLU, nn.GroupNorm, nn.LeakyReLU)):
        hooks.append(module.register_forward_hook(make_hook(name)))
    elif module.__class__.__name__ in {"ResidualBlock", "Downsample", "Upsample", "GroupNorm"}:
        hooks.append(module.register_forward_hook(make_hook(name)))

with torch.no_grad():
    latent = model.encoder(dummy)
    recon = model.decoder(latent)

for h in hooks:
    h.remove()

# Resumen compacto: entrada → encoder stages → latente → decoder → salida
summary = [
    {"stage": "input (MRI)", "shape": _shape(dummy), "numel": int(dummy.numel())},
    {"stage": "encoder → latent bottleneck", "shape": _shape(latent), "numel": int(latent.numel())},
    {"stage": "decoder → reconstruction", "shape": _shape(recon), "numel": int(recon.numel())},
]
print("=== Resumen AAE ===")
display(pd.DataFrame(summary))

print("\n=== Capas (activaciones en forward) ===")
df = pd.DataFrame(rows).drop_duplicates(subset=["stage"], keep="last")
display(df)

print("\n=== print(model) ===")
print(model)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nParámetros totales: {n_params:,}  |  entrenables: {n_trainable:,}")
print(f"Latente flatten (por sujeto): {int(latent[0].numel()):,}  (= {latent.shape[1:]} )")

Checkpoint: /workspace/IL-CLDM/result/exp_nm_s1_mri_rois_supcon/phase1_mri/checkpoints/MRI_AAE.pth.tar
epoch=10  selection_score=-1.9456464052200317
load_state_dict: missing=0 unexpected=0
config.latent_dim=1  device=cpu



=== Resumen AAE ===


,stage,shape,numel
0,input (MRI),"(1, 1, 108, 104, 104)",1168128
1,encoder → latent bottleneck,"(1, 1, 27, 26, 26)",18252
2,decoder → reconstruction,"(1, 1, 108, 104, 104)",1168128



=== Capas (activaciones en forward) ===


,stage,shape,numel
0,input,"(1, 1, 108, 104, 104)",1168128
1,encoder.model.0,"(1, 16, 108, 104, 104)",18690048
2,encoder.model.1.channel_up,"(1, 32, 108, 104, 104)",37380096
3,encoder.model.1.block.0.gn,"(1, 16, 108, 104, 104)",18690048
4,encoder.model.1.block.0,"(1, 16, 108, 104, 104)",18690048
...,...,...,...
81,decoder.model.6,"(1, 16, 108, 104, 104)",18690048
82,decoder.model.7.gn,"(1, 16, 108, 104, 104)",18690048
83,decoder.model.7,"(1, 16, 108, 104, 104)",18690048
84,decoder.model.8,"(1, 16, 108, 104, 104)",18690048



=== print(model) ===
AAE(
  (encoder): Encoder(
    (model): Sequential(
      (0): Conv3d(1, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (1): ResidualBlock(
        (block): Sequential(
          (0): GroupNorm(
            (gn): GroupNorm(16, 16, eps=1e-06, affine=True)
          )
          (1): ReLU()
          (2): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
          (3): GroupNorm(
            (gn): GroupNorm(16, 32, eps=1e-06, affine=True)
          )
          (4): ReLU()
          (5): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
        )
        (channel_up): Conv3d(16, 32, kernel_size=(1, 1, 1), stride=(1, 1, 1))
      )
      (2): Downsample(
        (conv): Conv3d(32, 32, kernel_size=(4, 4, 4), stride=(2, 2, 2), padding=(1, 1, 1))
      )
      (3): ResidualBlock(
        (block): Sequential(
          (0): GroupNorm(
            (gn): GroupNorm(16, 32, eps=1e-06, affine=True)
       